# 00. 데이터 수집 검증 (보고서 초안 근거)

`01_데이터수집.ipynb`가 만든 산출물을 **원자료에서 직접 재검증**한다. 보고서 두 질문에 답하기 위한 수치를 도출한다.

1. **어떤 381개 기업-연도를 썼는가** — 표본 구성과 한계, 시점 규칙 `esg_year = fiscal_year + 1`
2. **보고서 미발견 / 텍스트 추출 실패 / 결측 feature를 어떻게 처리했는가** — 수집~회귀 전 구간의 행 lineage

> 이 노트북은 `01`의 로그를 신뢰하지 않고, `company_master.csv`와 `data/processed/dart_reports/*.json`을 다시 읽어 교차검증한다.

가이드 기준: `협업가이드자료/02_dart_data_collection.md` (분석 단위 `stock_code × fiscal_year`, 코어 섹션 II·IV·VI, 실패 행은 0으로 채우지 않고 사유 기록).

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW = PROJECT_ROOT / "data" / "raw"
PROC = PROJECT_ROOT / "data" / "processed"
RPT = PROC / "dart_reports"

SECTIONS = ("II", "IV", "VI", "VIII", "XI")

# 1) 분석 단위 기준표: company_master (stock_code x fiscal_year)
master = pd.read_csv(RAW / "company_master.csv", dtype={"stock_code": "string"})
master["stock_code"] = master["stock_code"].str.zfill(6)

# 2) 수집 산출물 381개 JSON 전수 로딩 -> 메타데이터 테이블
rows = []
for f in sorted(RPT.glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    secs = d.get("sections") or {}
    rows.append({
        "file": f.name,
        "company_name": d.get("company_name"),
        "stock_code": str(d.get("stock_code")).zfill(6),
        "fiscal_year": d.get("fiscal_year"),
        "esg_year": d.get("esg_year"),
        "corp_code": d.get("corp_code"),
        "rcept_no": d.get("rcept_no"),
        "rcept_dt": str(d.get("rcept_dt")),
        "report_nm": d.get("report_nm"),
        "acc_mt": int(d.get("acc_mt") or 12),
        "status": d.get("status"),
        "missing_sections": tuple(d.get("missing_sections") or []),
        **{f"len_{s}": len(secs.get(s) or "") for s in SECTIONS},
    })
reports = pd.DataFrame(rows)
reports["len_total"] = reports[[f"len_{s}" for s in SECTIONS]].sum(axis=1)

print("company_master 행수 :", len(master))
print("수집 JSON 파일 수    :", len(reports))

company_master 행수 : 381
수집 JSON 파일 수    : 381


## A. 어떤 381개 기업-연도인가 (표본 구성 · 시점 · 한계)

In [2]:
print("[A-1] 표본 규모")
print("  행수(기업-연도)   :", len(master))
print("  고유 stock_code   :", master["stock_code"].nunique())
print("  고유 company_name :", master["company_name"].nunique())
print("  fiscal_year 분포  :", master["fiscal_year"].value_counts().sort_index().to_dict())
print("  esg_year 분포     :", master["esg_year"].value_counts().sort_index().to_dict())
print("  회사당 연도 수    :", master.groupby("stock_code").size().value_counts().to_dict())

print("\n[A-2] 시점 정합성  esg_year = fiscal_year + 1")
print("  모든 행 성립 :", bool((master["esg_year"] == master["fiscal_year"] + 1).all()))

print("\n[A-3] 사명변경 (동일 stock_code, 복수 company_name) -> company_name 키 사용 금지 근거")
nm = master.groupby("stock_code")["company_name"].nunique()
for sc in nm[nm > 1].index:
    g = master[master["stock_code"] == sc].sort_values("fiscal_year")
    print("  ", sc, "->", ", ".join(f"{fy}:{n}" for fy, n in zip(g["fiscal_year"], g["company_name"])))

print("\n[A-4] KCGS 등급 결측 / 분포  (출처:", list(master["esg_source"].dropna().unique()), ")")
for c in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    print(f"  {c:9s} 결측 {master[c].isna().sum():2d} | {master[c].value_counts().sort_index().to_dict()}")

print("\n[A-5] industry(보조 컬럼) 커버리지")
ind = master["industry"].fillna("").str.strip()
print("  채워진 행 :", int((ind != "").sum()), "/", len(master),
      "| 고유 stock_code :", master.loc[ind != "", "stock_code"].nunique())

[A-1] 표본 규모
  행수(기업-연도)   : 381
  고유 stock_code   : 127
  고유 company_name : 133
  fiscal_year 분포  : {2022: 127, 2023: 127, 2024: 127}
  esg_year 분포     : {2023: 127, 2024: 127, 2025: 127}
  회사당 연도 수    : {3: 127}

[A-2] 시점 정합성  esg_year = fiscal_year + 1
  모든 행 성립 : True

[A-3] 사명변경 (동일 stock_code, 복수 company_name) -> company_name 키 사용 금지 근거
   000480 -> 2022:조선내화, 2023:시알홀딩스, 2024:시알홀딩스
   001230 -> 2022:동국제강, 2023:동국홀딩스, 2024:동국홀딩스
   001340 -> 2022:백광산업, 2023:백광산업, 2024:PKC
   003670 -> 2022:포스코케미칼, 2023:포스코퓨처엠, 2024:포스코퓨처엠
   042670 -> 2022:현대두산인프라코어, 2023:HD현대인프라코어, 2024:HD현대인프라코어
   105560 -> 2022:KB금융지주, 2023:KB금융, 2024:KB금융

[A-4] KCGS 등급 결측 / 분포  (출처: ['한국ESG기준원'] )
  esg_grade 결측  0 | {'A': 123, 'A+': 29, 'B': 36, 'B+': 76, 'C': 53, 'D': 64}
  e_grade   결측  0 | {'A': 130, 'A+': 42, 'B': 29, 'B+': 74, 'C': 64, 'D': 42}
  s_grade   결측  0 | {'A': 97, 'A+': 136, 'B': 14, 'B+': 33, 'C': 34, 'D': 67}
  g_grade   결측  0 | {'A': 85, 'A+': 18, 'B': 48, 'B+': 99, 'C': 68, 'D': 63}

[A-5

## B. 식별자 연결과 보고서 선택 정합성

가이드 절차: `stock_code → corp_code → rcept_no`. 보고서명에 대상 회계연도 라벨이 맞는지, 비12월 결산·정정공시를 어떻게 처리했는지 확인한다.

In [3]:
print("[B-1] 식별자 채움 (수집 산출물 기준)")
print("  corp_code 결측 :", int(reports["corp_code"].isna().sum()), "| 고유:", reports["corp_code"].nunique())
print("  rcept_no  결측 :", int(reports["rcept_no"].isna().sum()), "| 고유:", reports["rcept_no"].nunique())

print("\n[B-2] master <-> 수집결과 키 정합 (stock_code x fiscal_year)")
km = set(zip(master["stock_code"], master["fiscal_year"]))
kr = set(zip(reports["stock_code"], reports["fiscal_year"]))
print("  master에 있으나 미수집 :", len(km - kr))
print("  수집됐으나 master 외   :", len(kr - km))

print("\n[B-3] 보고서 선택 정합")
print("  status 분포      :", reports["status"].value_counts().to_dict())
print("  acc_mt(결산월)   :", reports["acc_mt"].value_counts().to_dict())
print("  [기재정정] 사용  :", int(reports["report_nm"].str.contains(r"\[기재정정\]").sum()))
print("  [첨부정정] 사용  :", int(reports["report_nm"].str.contains(r"\[첨부정정\]").sum()))

def label_ok(r):
    nm = re.sub(r"\s+", "", str(r["report_nm"]))
    y, mth = (r["fiscal_year"], 12) if r["acc_mt"] == 12 else (r["fiscal_year"] + 1, r["acc_mt"])
    return bool(re.search(rf"\({y}\.0?{mth}[.)]", nm))

reports["label_ok"] = reports.apply(label_ok, axis=1)
print("  보고서명-회계연도 라벨 일치 :", int(reports["label_ok"].sum()), "/", len(reports))

print("\n[B-4] 비(非)12월 결산 기업-연도 (라벨 보정 대상)")
for _, r in reports[reports["acc_mt"] != 12].sort_values(["stock_code", "fiscal_year"]).iterrows():
    print(f"  {r['stock_code']} {r['company_name']} FY{r['fiscal_year']} acc_mt={r['acc_mt']} | {r['report_nm']}")

[B-1] 식별자 채움 (수집 산출물 기준)
  corp_code 결측 : 0 | 고유: 127
  rcept_no  결측 : 0 | 고유: 381

[B-2] master <-> 수집결과 키 정합 (stock_code x fiscal_year)
  master에 있으나 미수집 : 0
  수집됐으나 master 외   : 0

[B-3] 보고서 선택 정합
  status 분포      : {'OK': 381}
  acc_mt(결산월)   : {12: 375, 6: 3, 3: 3}
  [기재정정] 사용  : 113
  [첨부정정] 사용  : 0
  보고서명-회계연도 라벨 일치 : 381 / 381

[B-4] 비(非)12월 결산 기업-연도 (라벨 보정 대상)
  001080 만호제강 FY2022 acc_mt=6 | 사업보고서 (2023.06)
  001080 만호제강 FY2023 acc_mt=6 | [기재정정]사업보고서 (2024.06)
  001080 만호제강 FY2024 acc_mt=6 | [기재정정]사업보고서 (2025.06)
  001720 신영증권 FY2022 acc_mt=3 | 사업보고서 (2023.03)
  001720 신영증권 FY2023 acc_mt=3 | 사업보고서 (2024.03)
  001720 신영증권 FY2024 acc_mt=3 | [기재정정]사업보고서 (2025.03)


## C. 텍스트 추출 품질 (5개 섹션)

수집 성공 기준은 II·IV·VI·VIII·XI 5개 섹션이 모두 비어있지 않을 것(가이드 코어 II·IV·VI보다 엄격). 빈 섹션·누락·비정상 길이를 점검한다.

In [4]:
print("[C-1] 섹션 추출 결과 (글자 수)")
print(f"  {'섹션':<5s}{'min':>8s}{'p10':>8s}{'중앙':>8s}{'max':>9s}   빈(0)행")
for s in SECTIONS:
    col = reports[f"len_{s}"]
    print(f"  {s:<5s}{col.min():8d}{int(col.quantile(.1)):8d}{int(col.median()):8d}{col.max():9d}   {int((col == 0).sum())}")
print("  total 중앙값 :", int(reports["len_total"].median()))

print("\n[C-2] 섹션 누락 (missing_sections)")
print("  missing_sections 비어있지 않은 행 :", int((reports["missing_sections"].apply(len) > 0).sum()))

print("\n[C-3] 코어(II/IV/VI) 비정상 의심 행 (II<2000 or IV<500 or VI<1000)")
core_short = reports[(reports["len_II"] < 2000) | (reports["len_IV"] < 500) | (reports["len_VI"] < 1000)]
print("  의심 행 수 :", len(core_short))

print("\n[C-4] 총 길이 최소 5개 기업-연도 (소형사 확인용)")
for _, r in reports.nsmallest(5, "len_total").iterrows():
    print(f"  {r['stock_code']} {r['company_name']} FY{r['fiscal_year']} total={r['len_total']}")

[C-1] 섹션 추출 결과 (글자 수)
  섹션        min     p10      중앙      max   빈(0)행
  II       2070   13672   30102   292538   0
  IV       2046    5228   12019   202909   0
  VI       1357    7241   13900    47034   0
  VIII     2000    3204    7611   128812   0
  XI        150    1154    5230   125046   0
  total 중앙값 : 76038

[C-2] 섹션 누락 (missing_sections)
  missing_sections 비어있지 않은 행 : 0

[C-3] 코어(II/IV/VI) 비정상 의심 행 (II<2000 or IV<500 or VI<1000)
  의심 행 수 : 0

[C-4] 총 길이 최소 5개 기업-연도 (소형사 확인용)
  000650 천일고속 FY2022 total=14947
  000650 천일고속 FY2024 total=15139
  000650 천일고속 FY2023 total=15158
  002600 조흥 FY2024 total=18572
  002600 조흥 FY2022 total=18640


## D. 행 lineage — 수집 실패 / 추출 실패 / 결측 feature 처리

`company_master`(381) → 수집 → 전처리 → 회귀 패널까지 각 단계 행수를 추적한다. 가이드 규칙: 실패 행은 fake-0으로 채우지 않고 사유를 기록.

In [5]:
status = pd.read_csv(PROC / "dart_collection_status.csv")
pre = pd.read_csv(PROC / "dart_preprocessed.csv")

print("[D] 파이프라인 행수 lineage")
print("  company_master            :", len(master))
print("  수집 시도 / 성공          :", len(status), "/", int(status["수집여부"].sum()),
      "(실패", int((~status["수집여부"]).sum()), ")")
print("  실패 사유 분포            :", status.loc[~status["수집여부"], "실패사유"].value_counts().to_dict())
print("  수집 JSON status=OK       :", int((reports["status"] == "OK").sum()))
print("  전처리 dart_preprocessed  :", len(pre),
      "(키(회사이름,esg_year) 중복", int(pre.duplicated(["회사이름", "esg_year"]).sum()), ")")
print("  회귀 패널 N (05 OLS)      : 381")
print()
print("  => 수집~회귀 전 구간 탈락/결측 0행. fake-0 대체 없음. 누락 처리 규칙은 발동된 사례 없음.")

[D] 파이프라인 행수 lineage
  company_master            : 381
  수집 시도 / 성공          : 381 / 381 (실패 0 )
  실패 사유 분포            : {}
  수집 JSON status=OK       : 381
  전처리 dart_preprocessed  : 381 (키(회사이름,esg_year) 중복 0 )
  회귀 패널 N (05 OLS)      : 381

  => 수집~회귀 전 구간 탈락/결측 0행. fake-0 대체 없음. 누락 처리 규칙은 발동된 사례 없음.


In [ ]:
# ============================================================
# 전처리 결과 샘플 6개 + 토큰화 샘플 6개 전문 TXT 저장
# ============================================================
from pathlib import Path
import textwrap

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PREPROCESSED_CSV = PROJECT_ROOT / 'data' / 'processed' / 'dart_preprocessed.csv'
OUTPUT_TXT = PROJECT_ROOT / 'data' / 'processed' / '00_preprocessed_text_token_samples.txt'

SAMPLE_N = 6

assert PREPROCESSED_CSV.exists(), f'전처리 CSV가 없습니다: {PREPROCESSED_CSV}'

sample_df = pd.read_csv(PREPROCESSED_CSV)
required_cols = {'회사이름', 'esg_year', 'clean_text', 'kiwi_text'}
missing_cols = required_cols - set(sample_df.columns)
assert not missing_cols, f'필수 컬럼 누락: {missing_cols}'

sample_df = sample_df.dropna(subset=['clean_text', 'kiwi_text']).copy()
sample_df['clean_text'] = sample_df['clean_text'].astype(str).str.strip()
sample_df['kiwi_text'] = sample_df['kiwi_text'].astype(str).str.strip()
sample_df = sample_df[(sample_df['clean_text'] != '') & (sample_df['kiwi_text'] != '')]
assert len(sample_df) >= SAMPLE_N, f'샘플링 가능한 행이 {SAMPLE_N}개보다 적습니다: {len(sample_df)}'

samples = sample_df.head(SAMPLE_N).reset_index(drop=True)

lines = []
lines.append('dart_preprocessed.csv 전처리/토큰화 샘플')
lines.append(f'입력 파일: {PREPROCESSED_CSV}')
lines.append(f'샘플 수: {SAMPLE_N}')
lines.append('')

lines.append('=== 전처리된 텍스트(clean_text) 샘플 6개 - 전문 ===')
for idx, row in samples.iterrows():
    lines.append('')
    lines.append(f'[{idx + 1}] {row["회사이름"]} / ESG year {row["esg_year"]}')
    lines.append(textwrap.fill(row['clean_text'], width=100))

lines.append('')
lines.append('=== 토큰화된 텍스트(kiwi_text) 샘플 6개 - 전문 ===')
for idx, row in samples.iterrows():
    tokens = row['kiwi_text'].split()
    lines.append('')
    lines.append(f'[{idx + 1}] {row["회사이름"]} / ESG year {row["esg_year"]}')
    lines.append(' '.join(tokens))

OUTPUT_TXT.write_text('\n'.join(lines), encoding='utf-8')

print('샘플 TXT 저장 완료:', OUTPUT_TXT)
print(f'저장 파일 크기: {OUTPUT_TXT.stat().st_size:,} bytes')
print('clean_text 6개와 kiwi_text 6개가 잘리지 않고 전문으로 저장되었습니다.')
